# GEPA Showcase — Hard Reasoning on Logic Puzzles

GEPA's reflective prompt mutation is most useful on tasks where the LLM has a clear failure mode and benefits from learning a structured strategy. Einstein-style logic puzzles are a perfect test case: each puzzle has a unique solution that follows from the clues by deduction, but a naive prompt almost always misassigns at least one cell.

**Dataset:** `logic_puzzles.json` — 100 4×4 constraint-satisfaction puzzles, each with a verified-unique solution and a minimal clue set (synthetic, fully permissive).

**What you'll learn:**
1. Encode a structured-output puzzle as a single signature
2. Write a composite metric: per-cell accuracy + full-solution bonus
3. Watch GEPA's reflection translate metric feedback into a sharper prompt
4. Compare a baseline ChainOfThought vs. the GEPA-optimized program

**Prerequisites:** Complete [01_quickstart.ipynb](01_quickstart.ipynb).

In [ ]:
import json
import os
from pathlib import Path

import dspy

from skynet_client import DSPyServiceClient, JobMonitor, serialize_source

In [ ]:
BASE_URL = os.getenv("DSPY_SERVICE_URL", "http://localhost:8000")
LM_BASE_URL = os.getenv("DSPY_LM_BASE_URL", "https://api.openai.com/v1")

# LM_API_KEY is the canonical name; OPENAI_API_KEY is accepted as a
# fallback for backwards-compat and the public OpenAI dev path.
LM_API_KEY = os.getenv("LM_API_KEY") or os.getenv("OPENAI_API_KEY")
if not LM_API_KEY:
    raise ValueError("Set LM_API_KEY (or OPENAI_API_KEY) — any non-empty token works for self-hosted gateways.")

MODEL_CONFIG = {
    "name": "openai/gpt-4o-mini",
    "base_url": LM_BASE_URL,
    "model_type": "responses",
    "temperature": 1.0,
    "max_tokens": 20000,
    "extra": {"api_key": LM_API_KEY},
}

dspy.configure(
    lm=dspy.LM(
        "openai/gpt-4o-mini",
        model_type="responses",
        temperature=1.0,
        max_tokens=20000,
        api_base=LM_BASE_URL,
        api_key=LM_API_KEY,
    )
)

client = DSPyServiceClient(BASE_URL)
client.health()

## 1. Load & Inspect a Puzzle

Each row gives entities, attribute pools, and a clue list. The solution is a nested dict mapping each entity to its full attribute assignment.

In [ ]:
with open(Path("../../data/logic_puzzles.json")) as f:
    raw_dataset = json.load(f)

print(f"{len(raw_dataset)} puzzles")
p = raw_dataset[0]
print(f"\nEntities:    {p['entities']}")
print(f"Attributes:  {p['attributes']}")
print(f"Clue count:  {len(p['clues'])}")
print(f"Sample clues:")
for c in p["clues"][:3]:
    print(f"  - {c}")
print(f"\nSolution:")
for ent, attrs in p["solution"].items():
    print(f"  {ent}: {attrs}")

## 2. Flatten Inputs for the LLM

DSPy signatures want flat strings, not nested objects. Stringify the puzzle into one prompt-ready field; serialise the solution as JSON for round-trippable comparison.

In [ ]:
def serialise_puzzle(row):
    """Render one puzzle as a single string the LLM can consume."""
    lines = []
    lines.append(f"Entities: {', '.join(row['entities'])}")
    lines.append("Attribute pools:")
    for attr, values in row["attributes"].items():
        lines.append(f"  - {attr}: {', '.join(values)}")
    lines.append("Clues:")
    for c in row["clues"]:
        lines.append(f"  - {c}")
    return "\n".join(lines)


dataset = []
for row in raw_dataset:
    dataset.append({
        "puzzle": serialise_puzzle(row),
        "solution_json": json.dumps(row["solution"], sort_keys=True),
    })

print(dataset[0]["puzzle"])
print()
print(f"Gold solution_json: {dataset[0]['solution_json'][:120]}...")

## 3. Signature

A single string in, a single JSON string out. The LLM has to do the deduction and emit a complete assignment.

In [ ]:
class LogicPuzzleSolver(dspy.Signature):
    """Solve a 4-position constraint-satisfaction puzzle from clues and emit a JSON solution."""
    puzzle: str = dspy.InputField(desc="Entities, attribute pools, and clues for one puzzle")
    solution_json: str = dspy.OutputField(
        desc='JSON object mapping each entity (e.g. "position_1") to its attribute assignment, '
             'e.g. {"position_1": {"color": "red", "pet": "cat"}}'
    )


SIGNATURE_CODE = serialize_source(LogicPuzzleSolver)
print(SIGNATURE_CODE)

## 4. Composite Metric

Three signals make GEPA's feedback useful here:
1. **Validity** — does the prediction parse as JSON with the expected entity keys?
2. **Per-cell accuracy** — fraction of (entity, attribute) cells that match gold.
3. **Full-solution bonus** — was every cell correct?

The feedback string explicitly names which cells disagreed so GEPA's reflection model knows what to fix.

In [ ]:
METRIC_CODE = '''
def logic_puzzle_metric(gold, pred, trace=None, pred_name=None, pred_trace=None):
    import json as _json

    try:
        gold_sol = _json.loads(gold.solution_json)
    except Exception:
        return dspy.Prediction(score=0.0, feedback="Gold solution unparsable.")

    try:
        pred_sol = _json.loads(pred.solution_json)
    except Exception:
        return dspy.Prediction(score=0.0, feedback="Output is not valid JSON. Emit a JSON object only — no prose.")

    if not isinstance(pred_sol, dict):
        return dspy.Prediction(score=0.0, feedback="Output JSON is not an object.")

    total = 0
    correct = 0
    misses = []
    for entity, attrs in gold_sol.items():
        pred_attrs = pred_sol.get(entity) or {}
        if not isinstance(pred_attrs, dict):
            misses.append(f"{entity}: expected dict, got {type(pred_attrs).__name__}")
            total += len(attrs)
            continue
        for attr, gold_val in attrs.items():
            total += 1
            pv = pred_attrs.get(attr)
            if str(pv).strip().lower() == str(gold_val).strip().lower():
                correct += 1
            else:
                misses.append(f"{entity}.{attr}: expected {gold_val!r}, got {pv!r}")

    if total == 0:
        return dspy.Prediction(score=0.0, feedback="Gold solution had no cells.")

    cell_accuracy = correct / total
    bonus = 0.2 if correct == total else 0.0
    score = round(min(1.0, cell_accuracy * 0.8 + bonus + (0.0 if correct < total else 0.0)), 3)

    if correct == total:
        return dspy.Prediction(score=1.0, feedback="All cells correct.")

    head = misses[:5]
    feedback = (
        f"Got {correct}/{total} cells. Mismatches: " + "; ".join(head) +
        ("; ..." if len(misses) > 5 else "") +
        ". Re-derive the full assignment from the clues; do not guess."
    )
    return dspy.Prediction(score=score, feedback=feedback)
'''

## 5. Submit Optimization

In [ ]:
payload = {
    "username": "notebook-user",
    "module_name": "dspy.ChainOfThought",
    "signature_code": SIGNATURE_CODE,
    "metric_code": METRIC_CODE,
    "optimizer_name": "dspy.GEPA",
    "optimizer_kwargs": {"auto": "medium", "num_threads": 8},
    "compile_kwargs": {},
    "dataset": dataset,
    "column_mapping": {
        "inputs": {"puzzle": "puzzle"},
        "outputs": {"solution_json": "solution_json"},
    },
    "split_fractions": {"train": 0.5, "val": 0.3, "test": 0.2},
    "shuffle": True,
    "seed": 42,
    "model_config": MODEL_CONFIG,
    "reflection_model_config": MODEL_CONFIG,
}

optimization_id = client.submit(payload)
print(f"Submitted: {optimization_id}")

In [ ]:
monitor = JobMonitor(client, optimization_id)
result = monitor.poll(interval=10)

if result["status"] == "success":
    r = result["result"]
    print(f"\nBaseline cell-accuracy:  {r.get('baseline_test_metric', 'N/A')}")
    print(f"Optimized cell-accuracy: {r.get('optimized_test_metric', 'N/A')}")
    print(f"Runtime: {r.get('runtime_seconds', 0):.0f}s")
    print(f"LM calls during search: {r.get('num_lm_calls', 'N/A')}")
else:
    print(f"\nFailed: {result.get('message')}")

## 6. Inspect the GEPA-Discovered Prompt

GEPA mutates the signature's instruction string based on metric feedback. Loading the optimized program lets you read what it learned.

In [ ]:
program = client.load_program(optimization_id)
predictor = next(iter(program.predictors()))
print("Optimized instruction:\n")
print(predictor.signature.instructions)

## 7. Solve a Held-Out Puzzle

In [ ]:
unseen = raw_dataset[-1]
prompt = serialise_puzzle(unseen)

response = program(puzzle=prompt)
print("Predicted solution:")
print(response.solution_json)
print()
print("Gold solution:")
print(json.dumps(unseen["solution"], indent=2, sort_keys=True))

## 8. Per-Example Score Curve

On a hard reasoning task, GEPA usually lifts the median test score and flips a handful of previously-failing examples into wins. Plot the per-example deltas to see where the gain came from.

In [ ]:
tr = client.test_results(optimization_id)
baseline = tr.get("baseline", [])
optimized = tr.get("optimized", [])

flips = sum(1 for b, o in zip(baseline, optimized, strict=True)
            if (o.get("score") or 0) > (b.get("score") or 0))
regressions = sum(1 for b, o in zip(baseline, optimized, strict=True)
                  if (o.get("score") or 0) < (b.get("score") or 0))
print(f"Improved on {flips} test examples; regressed on {regressions}.")

## What's Next

- Tune `optimizer_kwargs={"auto": "heavy"}` on a stronger reflection model to push the cell-accuracy ceiling.
- Increase puzzle size (5×5, 6×6) by editing `scripts/data/generate_logic_puzzles.py` — uniqueness is verified by construction.
- **API Reference** — `http://localhost:8000/reference`